# Cleaning Dataset Hortikultura — Kabupaten Kediri

Notebook ini membersihkan 6 file hortikultura secara **spesifik per file** (bukan generik), karena tiap file
punya struktur yang beda. Ditulis berdasarkan isi asli tiap file, bukan tebakan.

**File yang dibersihkan:**
| File | Struktur asli |
|---|---|
| `Luas_Panen_Hortikultura_Sayur_per_kecamatan.csv` | Wide: 1 baris = 1 kecamatan, kolom = tiap komoditas. Ada baris total "Kediri" |
| `Produksi_Hortikultura_Buah_per_kecamatan.csv` | Sama (wide, ada baris "Kediri") |
| `Produksi_Hortikultura_Sayur_per_kecamatan.csv` | Sama (wide, ada baris "Kediri") |
| `Luas_Panen_Hortikultura_Sayur.csv` | Long, tingkat **Kabupaten** saja (kolom: Jenis Tanaman, nilai) |
| `Total_Produksi_Hortikultura_Buah.csv` | Sama (long, tingkat Kabupaten) |
| `Total_Produksi_Hortikultura_Sayur.csv` | Sama (long, tingkat Kabupaten) |

File biofarmaka tidak diproses di notebook ini.

**Temuan dari pengecekan isi file:**
1. **Datanya bukan estimasi.** Baris "Kediri" di tiap file wide adalah total resmi, dan **sama persis** dengan
   jumlah 26 kecamatan. Ini dipakai sebagai validasi otomatis.
2. **Satuan tidak seragam** di file luas panen sayur: mayoritas **hektar**, jamur dalam **m²**. Kolom `satuan`
   disimpan eksplisit, tidak dipaksa jadi satu satuan.
3. **Label satuan Jamur tiram salah di sumber** (teks "meter persegi", kode "Ha"). **Dikoreksi ke M2** karena
   file kabupaten menulis "(m2)" dengan nilai yang sama (2948) dan dua kolom jamur lain berkode M2. Koreksi
   dicetak dan diverifikasi otomatis (cell 7 dan 10), tidak dilakukan diam-diam.
4. **Penulisan satuan diseragamkan** (`ha`/`Ha` jadi `Ha`, `m2`/`M2` jadi `M2`, `kw` dan `kw/qui` jadi `Kw`)
   supaya file per kecamatan dan file kabupaten konsisten.
5. **Nenas di Ngancar sangat dominan** (±96% nenas kabupaten dan ±61% seluruh produksi buah). Sudah dicek: sesuai
   file sumber. Satuan produksi buah adalah **kuintal**, bukan ton (ton = kuintal / 10). Lihat cell 11.
6. **Nama kecamatan "Kayen Kidul"** (pakai spasi) disamakan ke "Kayenkidul".

**Catatan soal "-" (strip):** dalam data BPS, "-" biasanya berarti **tidak ada produksi/luas panen**, sehingga
di sini diperlakukan sebagai **0**. Beda dengan `...` yang diperlakukan sebagai data tidak tersedia (NaN).

**Output** (`data/clean/hortikultura/`):
- `luas_panen_hortikultura_sayur_per_kecamatan.csv` → `kecamatan, komoditas, luas_panen, satuan`
- `produksi_hortikultura_buah_per_kecamatan.csv` → `kecamatan, komoditas, produksi_kuintal`
- `produksi_hortikultura_sayur_per_kecamatan.csv` → `kecamatan, komoditas, produksi_kuintal`
- `luas_panen_hortikultura_sayur_kabupaten.csv` → `komoditas, luas_panen, satuan`
- `total_produksi_hortikultura_buah_kabupaten.csv` → `komoditas, produksi_kuintal, satuan`
- `total_produksi_hortikultura_sayur_kabupaten.csv` → `komoditas, produksi_kuintal, satuan`


In [1]:
# 1. IMPORT LIBRARY

import os
import re

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", None)
print("✓ Library siap.")

✓ Library siap.


In [2]:
# 2. CARI LOKASI PROJECT (folder 'data') SECARA OTOMATIS

CURRENT_DIR = os.getcwd()
PROJECT_ROOT = None

search_dir = CURRENT_DIR
while True:
    if os.path.isdir(os.path.join(search_dir, "data")):
        PROJECT_ROOT = search_dir
        break
    parent_dir = os.path.dirname(search_dir)
    if parent_dir == search_dir:
        break
    search_dir = parent_dir

if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "Folder 'data' tidak ditemukan dari direktori notebook ini ke atas. "
        "Pastikan notebook dijalankan dari dalam repository (mis. dari folder 'notebooks/')."
    )

DATA_DIR = os.path.join(PROJECT_ROOT, "data")
RAW_HORTI = os.path.join(DATA_DIR, "raw", "hortikultura")
OUT_HORTI = os.path.join(DATA_DIR, "clean", "hortikultura")

os.makedirs(RAW_HORTI, exist_ok=True)
os.makedirs(OUT_HORTI, exist_ok=True)

print("Project root:", PROJECT_ROOT)

Project root: d:\SDT\Project Analitika Data\AgriDash-0.1


In [3]:
# 3. MASTER KECAMATAN (26 kecamatan baku) & KOREKSI NAMA

MASTER_KECAMATAN = [
    "Mojo", "Semen", "Ngadiluwih", "Kras", "Ringinrejo", "Kandat", "Wates", "Ngancar",
    "Plosoklaten", "Gurah", "Puncu", "Kepung", "Kandangan", "Pare", "Badas", "Kunjang",
    "Plemahan", "Purwoasri", "Papar", "Pagu", "Kayenkidul", "Gampengrejo", "Ngasem",
    "Banyakan", "Grogol", "Tarokan",
]

# Perbaikan nama yang ditemukan langsung di file (bukan tebakan/fuzzy, supaya aman & presisi).
KOREKSI_KECAMATAN = {
    "Kayen Kidul": "Kayenkidul",
}

print(f"✓ {len(MASTER_KECAMATAN)} kecamatan baku siap.")

✓ 26 kecamatan baku siap.


In [4]:
# 4. FUNGSI BANTU

def parse_angka(nilai):
    """'-' -> 0.0 (konvensi BPS: tidak ada produksi/luas panen).
    '...' atau kosong -> NaN (data tidak valid/tidak tersedia, BUKAN nol)."""
    if pd.isna(nilai):
        return np.nan
    s = str(nilai).strip()
    if s == "-":
        return 0.0
    if s in ("", "...", "nan"):
        return np.nan
    s = s.replace(",", "")
    try:
        return float(s)
    except ValueError:
        return np.nan


def standar_kecamatan(nama):
    nama = str(nama).strip()
    return KOREKSI_KECAMATAN.get(nama, nama)


# Penulisan satuan diseragamkan supaya file per kecamatan dan file kabupaten bisa digabung/dibandingkan.
# 'kw' dan 'kw/qui' sama-sama kuintal; 'ha' dan 'm2' beda huruf besar/kecil antar file.
SATUAN_BAKU = {"ha": "Ha", "m2": "M2", "kw": "Kw", "kw/qui": "Kw", "pohon": "Pohon"}


def baku_satuan(satuan):
    if satuan is None or pd.isna(satuan):
        return None
    s = str(satuan).strip()
    return SATUAN_BAKU.get(s.lower(), s)


POLA_KOLOM_WIDE = re.compile(r"^(?:Luas Panen|Produksi)\s+(.*?)\s*\(([^()]*)\)\s*\(([^()]*)\)$")


def urai_kolom_wide(nama_kolom):
    """'Luas Panen Jahe (meter persegi) (M2)' -> ('Jahe', 'meter persegi', 'M2').
    Return (None, None, None) kalau pola tidak cocok (supaya ketahuan, bukan diam-diam dilewati)."""
    m = POLA_KOLOM_WIDE.match(nama_kolom.strip())
    if not m:
        return None, None, None
    return m.group(1).strip(), m.group(2).strip(), m.group(3).strip()


def urai_baris_kabupaten(teks):
    """'Bawang Daun/Scallion(ha)' -> ('Bawang Daun/Scallion', 'ha'). Menangani teks yang ada
    baris baru di dalamnya (field CSV multi-baris yang dikutip)."""
    teks = re.sub(r"\s+", " ", str(teks).strip())
    m = re.match(r"^(.*)\(([^()]+)\)$", teks)
    if not m:
        return teks, None
    return m.group(1).strip(), m.group(2).strip()


print("✓ Fungsi bantu siap.")

✓ Fungsi bantu siap.


In [5]:
# 5. FUNGSI: BERSIHKAN SATU FILE WIDE (kecamatan x komoditas, ada baris total 'Kediri')

def bersihkan_file_wide(path, nama_value, cek_satuan_konsisten=None, koreksi_satuan=None):
    """nama_value: nama kolom nilai di output ('luas_panen' atau 'produksi_kuintal').
    cek_satuan_konsisten: satuan yang diharapkan sama untuk semua komoditas (mis. 'Kw'), atau
    None kalau satuan memang boleh campur (maka kolom 'satuan' ikut disimpan).
    koreksi_satuan: dict {nama komoditas huruf kecil: kode satuan benar} untuk salah label yang
    sudah dibuktikan (lihat cell yang memanggilnya). Setiap koreksi selalu dicetak, tidak diam-diam."""

    df_raw = pd.read_csv(path, encoding="utf-8-sig")
    kolom_kecamatan = df_raw.columns[0]
    kolom_komoditas = [c for c in df_raw.columns[1:]]

    print(f"--- {os.path.basename(path)} ---")
    print(f"  {len(df_raw)} baris (termasuk baris total), {len(kolom_komoditas)} kolom komoditas.")

    # Pisahkan baris total Kabupaten ('Kediri') dari baris kecamatan.
    is_total = df_raw[kolom_kecamatan].astype(str).str.strip().eq("Kediri")
    if not is_total.any():
        print("  ⚠️  Baris total 'Kediri' tidak ditemukan — validasi sum dilewati.")
        baris_total = None
    else:
        baris_total = df_raw.loc[is_total].iloc[0]
    df_kec = df_raw.loc[~is_total].copy()

    df_kec["kecamatan"] = df_kec[kolom_kecamatan].apply(standar_kecamatan)
    jumlah_unik = df_kec["kecamatan"].nunique()
    if jumlah_unik != len(MASTER_KECAMATAN):
        print(f"  ⚠️  Kecamatan unik = {jumlah_unik}, bukan {len(MASTER_KECAMATAN)}.")
    tidak_dikenal = set(df_kec["kecamatan"]) - set(MASTER_KECAMATAN)
    if tidak_dikenal:
        print(f"  ⚠️  Nama kecamatan tidak dikenali: {tidak_dikenal}")

    baris_keluar = []
    kolom_dibuang = []

    for kolom in kolom_komoditas:
        nama_komoditas, satuan_teks, satuan_kode = urai_kolom_wide(kolom)
        if nama_komoditas is None:
            print(f"  ⚠️  Nama kolom tidak dikenali polanya, dilewati: {kolom!r}")
            kolom_dibuang.append((kolom, "pola nama kolom tidak dikenali"))
            continue

        satuan_kode = baku_satuan(satuan_kode)
        if koreksi_satuan and nama_komoditas.lower() in koreksi_satuan:
            satuan_baru = koreksi_satuan[nama_komoditas.lower()]
            if satuan_baru != satuan_kode:
                print(f"  ℹ️  Satuan '{nama_komoditas}' dikoreksi: {satuan_kode!r} -> {satuan_baru!r}.")
                satuan_kode = satuan_baru

        nilai_mentah = df_kec[kolom].astype(str)
        proporsi_invalid = nilai_mentah.eq("...").mean()
        if proporsi_invalid == 1.0:
            print(f"  ⚠️  Kolom '{nama_komoditas}' isinya '...' di semua baris (bukan data). Dibuang dari output.")
            kolom_dibuang.append((kolom, "seluruh nilai '...' (data tidak valid)"))
            continue
        elif proporsi_invalid > 0:
            print(f"  ⚠️  Kolom '{nama_komoditas}': {proporsi_invalid:.0%} baris berisi '...' (data tidak valid sebagian).")

        if cek_satuan_konsisten and satuan_kode != cek_satuan_konsisten:
            print(f"  ⚠️  Satuan '{nama_komoditas}' = {satuan_kode!r}, berbeda dari yang diharapkan ({cek_satuan_konsisten!r}).")

        satuan_m2_kata = "meter" in satuan_teks.lower()
        satuan_kode_m2 = satuan_kode.upper() == "M2"
        if satuan_m2_kata != satuan_kode_m2 and satuan_kode.lower() not in ("kw",):
            print(f"  ⚠️  Label satuan tidak konsisten untuk '{nama_komoditas}': teks={satuan_teks!r}, kode={satuan_kode!r}.")

        for _, baris in df_kec.iterrows():
            rec = {"kecamatan": baris["kecamatan"], "komoditas": nama_komoditas,
                  nama_value: parse_angka(baris[kolom])}
            if cek_satuan_konsisten is None:
                rec["satuan"] = satuan_kode
            baris_keluar.append(rec)

    df_long = pd.DataFrame(baris_keluar)

    # Validasi: total per komoditas (sum 26 kecamatan) vs baris 'Kediri'.
    if baris_total is not None:
        cek = []
        for kolom in kolom_komoditas:
            nama_komoditas, _, _ = urai_kolom_wide(kolom)
            if nama_komoditas is None or any(nama_komoditas == k for k, _ in kolom_dibuang):
                continue
            total_resmi = parse_angka(baris_total[kolom])
            total_hitung = df_long.loc[df_long.komoditas == nama_komoditas, nama_value].sum()
            if pd.notna(total_resmi) and not np.isclose(total_resmi, total_hitung, atol=0.01):
                cek.append((nama_komoditas, total_resmi, total_hitung))
        if cek:
            print(f"  ⚠️  {len(cek)} komoditas yang totalnya TIDAK cocok dengan baris 'Kediri':")
            for nk, tr, th in cek:
                print(f"      {nk}: resmi={tr}, hasil hitung={th}")
        else:
            print("  ✓ Semua total per komoditas cocok dengan baris 'Kediri'.")

    print(f"  ✓ Hasil akhir: {len(df_long)} baris ({df_long['komoditas'].nunique()} komoditas × {jumlah_unik} kecamatan).\n")
    return df_long


print("✓ Fungsi bersihkan_file_wide siap.")

✓ Fungsi bersihkan_file_wide siap.


In [6]:
# 6. FUNGSI: BERSIHKAN FILE LONG TINGKAT KABUPATEN (Jenis Tanaman, nilai)

def bersihkan_file_kabupaten(path, nama_value):
    df_raw = pd.read_csv(path, encoding="utf-8-sig")
    kolom_nama, kolom_nilai = df_raw.columns[0], df_raw.columns[1]

    print(f"--- {os.path.basename(path)} ---")
    print(f"  {len(df_raw)} baris komoditas.")

    hasil = []
    for _, baris in df_raw.iterrows():
        nama_komoditas, satuan = urai_baris_kabupaten(baris[kolom_nama])
        satuan = baku_satuan(satuan)
        rec = {"komoditas": nama_komoditas, nama_value: parse_angka(baris[kolom_nilai])}
        if satuan is not None:
            rec["satuan"] = satuan
        hasil.append(rec)

    df_out = pd.DataFrame(hasil)
    satuan_unik = df_out["satuan"].nunique() if "satuan" in df_out else 0
    if satuan_unik > 1:
        print(f"  ⚠️  Satuan tidak seragam: {sorted(df_out['satuan'].unique())}")

    print(f"  ✓ Hasil akhir: {len(df_out)} baris.\n")
    return df_out


print("✓ Fungsi bersihkan_file_kabupaten siap.")

✓ Fungsi bersihkan_file_kabupaten siap.


In [7]:
# 7. HORTIKULTURA SAYUR — Luas Panen per Kecamatan

path = os.path.join(RAW_HORTI, "Luas_Panen_Hortikultura_Sayur_per_kecamatan.csv")
# Koreksi satuan Jamur tiram: Ha -> M2. Dasarnya (bukan tebakan):
#  1) header kolom menulis "meter persegi", hanya kode singkatannya yang "Ha" (salah ketik di sumber)
#  2) file kabupaten menulis Jamur Tiram "(m2)" dengan nilai sama (2948), dicek otomatis di cell 10
#  3) dua kolom jamur lain di file yang sama (Merang, Lainnya) berkode M2
KOREKSI_SATUAN_LP_SAYUR = {"jamur tiram": "M2"}

df_lp_sayur_kec = bersihkan_file_wide(path, nama_value="luas_panen", cek_satuan_konsisten=None,
                                      koreksi_satuan=KOREKSI_SATUAN_LP_SAYUR)

display(df_lp_sayur_kec.head(10))
df_lp_sayur_kec.to_csv(os.path.join(OUT_HORTI, "luas_panen_hortikultura_sayur_per_kecamatan.csv"),
                       index=False, encoding="utf-8-sig")

--- Luas_Panen_Hortikultura_Sayur_per_kecamatan.csv ---
  27 baris (termasuk baris total), 26 kolom komoditas.
  ℹ️  Satuan 'Jamur tiram' dikoreksi: 'Ha' -> 'M2'.
  ✓ Semua total per komoditas cocok dengan baris 'Kediri'.
  ✓ Hasil akhir: 676 baris (26 komoditas × 26 kecamatan).



,kecamatan,komoditas,luas_panen,satuan
0,Mojo,Bawang Daun,0.0,Ha
1,Semen,Bawang Daun,0.0,Ha
2,Ngadiluwih,Bawang Daun,4.0,Ha
3,Kras,Bawang Daun,0.0,Ha
4,Ringinrejo,Bawang Daun,0.0,Ha
5,Kandat,Bawang Daun,10.5,Ha
6,Wates,Bawang Daun,0.0,Ha
7,Ngancar,Bawang Daun,0.0,Ha
8,Plosoklaten,Bawang Daun,0.0,Ha
9,Gurah,Bawang Daun,0.0,Ha


In [8]:
# 8. HORTIKULTURA BUAH — Produksi per Kecamatan (satuan seragam: kuintal)

path = os.path.join(RAW_HORTI, "Produksi_Hortikultura_Buah_per_kecamatan.csv")
df_produksi_buah_kec = bersihkan_file_wide(path, nama_value="produksi_kuintal", cek_satuan_konsisten="Kw")

display(df_produksi_buah_kec.head(10))
df_produksi_buah_kec.to_csv(os.path.join(OUT_HORTI, "produksi_hortikultura_buah_per_kecamatan.csv"),
                            index=False, encoding="utf-8-sig")

--- Produksi_Hortikultura_Buah_per_kecamatan.csv ---
  27 baris (termasuk baris total), 27 kolom komoditas.
  ✓ Semua total per komoditas cocok dengan baris 'Kediri'.
  ✓ Hasil akhir: 702 baris (27 komoditas × 26 kecamatan).



,kecamatan,komoditas,produksi_kuintal
0,Mojo,Alpukat,24632.0
1,Semen,Alpukat,8200.0
2,Ngadiluwih,Alpukat,3604.0
3,Kras,Alpukat,3540.0
4,Ringinrejo,Alpukat,12272.0
5,Kandat,Alpukat,3240.0
6,Wates,Alpukat,14634.0
7,Ngancar,Alpukat,32580.0
8,Plosoklaten,Alpukat,6875.0
9,Gurah,Alpukat,1343.0


In [9]:
# 9. HORTIKULTURA SAYUR — Produksi per Kecamatan (satuan seragam: kuintal)

path = os.path.join(RAW_HORTI, "Produksi_Hortikultura_Sayur_per_kecamatan.csv")
df_produksi_sayur_kec = bersihkan_file_wide(path, nama_value="produksi_kuintal", cek_satuan_konsisten="Kw")

display(df_produksi_sayur_kec.head(10))
df_produksi_sayur_kec.to_csv(os.path.join(OUT_HORTI, "produksi_hortikultura_sayur_per_kecamatan.csv"),
                             index=False, encoding="utf-8-sig")

--- Produksi_Hortikultura_Sayur_per_kecamatan.csv ---
  27 baris (termasuk baris total), 26 kolom komoditas.
  ✓ Semua total per komoditas cocok dengan baris 'Kediri'.
  ✓ Hasil akhir: 676 baris (26 komoditas × 26 kecamatan).



,kecamatan,komoditas,produksi_kuintal
0,Mojo,Bawang Daun,0.0
1,Semen,Bawang Daun,0.0
2,Ngadiluwih,Bawang Daun,290.0
3,Kras,Bawang Daun,0.0
4,Ringinrejo,Bawang Daun,0.0
5,Kandat,Bawang Daun,909.0
6,Wates,Bawang Daun,0.0
7,Ngancar,Bawang Daun,0.0
8,Plosoklaten,Bawang Daun,0.0
9,Gurah,Bawang Daun,0.0


In [10]:
# 10. FILE TINGKAT KABUPATEN (tidak per kecamatan)

path = os.path.join(RAW_HORTI, "Luas_Panen_Hortikultura_Sayur.csv")
df_lp_sayur_kab = bersihkan_file_kabupaten(path, nama_value="luas_panen")
display(df_lp_sayur_kab.head(10))
df_lp_sayur_kab.to_csv(os.path.join(OUT_HORTI, "luas_panen_hortikultura_sayur_kabupaten.csv"),
                       index=False, encoding="utf-8-sig")

# Verifikasi dasar koreksi satuan Jamur tiram (cell 7): jumlah 26 kecamatan harus sama dengan nilai
# kabupaten, dan satuan di file kabupaten harus m2.
jt_kec = df_lp_sayur_kec[df_lp_sayur_kec["komoditas"].str.lower() == "jamur tiram"]
jt_kab = df_lp_sayur_kab[df_lp_sayur_kab["komoditas"].str.lower().str.startswith("jamur tiram")]
if len(jt_kec) and len(jt_kab):
    nilai_sama = np.isclose(jt_kec["luas_panen"].sum(), jt_kab["luas_panen"].iloc[0])
    satuan_sama = str(jt_kab["satuan"].iloc[0]).lower() == str(jt_kec["satuan"].iloc[0]).lower()
    if nilai_sama and satuan_sama:
        print(f"✓ Koreksi satuan Jamur tiram terverifikasi: jumlah kecamatan = nilai kabupaten = "
              f"{jt_kab['luas_panen'].iloc[0]:,.0f} {jt_kec['satuan'].iloc[0]}.")
    else:
        print("⚠️  Koreksi satuan Jamur tiram TIDAK terverifikasi oleh file kabupaten. Cek ulang ke sumber BPS.")

path = os.path.join(RAW_HORTI, "Total_Produksi_Hortikultura_Buah.csv")
df_produksi_buah_kab = bersihkan_file_kabupaten(path, nama_value="produksi_kuintal")
display(df_produksi_buah_kab.head(10))
df_produksi_buah_kab.to_csv(os.path.join(OUT_HORTI, "total_produksi_hortikultura_buah_kabupaten.csv"),
                            index=False, encoding="utf-8-sig")

path = os.path.join(RAW_HORTI, "Total_Produksi_Hortikultura_Sayur.csv")
df_produksi_sayur_kab = bersihkan_file_kabupaten(path, nama_value="produksi_kuintal")
display(df_produksi_sayur_kab.head(10))
df_produksi_sayur_kab.to_csv(os.path.join(OUT_HORTI, "total_produksi_hortikultura_sayur_kabupaten.csv"),
                             index=False, encoding="utf-8-sig")

--- Luas_Panen_Hortikultura_Sayur.csv ---
  26 baris komoditas.
  ⚠️  Satuan tidak seragam: ['Ha', 'M2']
  ✓ Hasil akhir: 26 baris.



,komoditas,luas_panen,satuan
0,Bawang Daun/Scallion,610.5,Ha
1,Bawang Merah/Shallots,1205.5,Ha
2,Bawang Putih/Garlic,0.0,Ha
3,Bayam/Spinach,45.0,Ha
4,Buncis/string bean,167.5,Ha
5,Cabai Rawit/Chili/Cayenne Pepper,9761.0,Ha
6,Kacang Panjang/Long Beans,419.7,Ha
7,Kangkung/Water Spinach,15.0,Ha
8,Kembang Kol/Cauliflower,190.0,Ha
9,Kentang/Potato,0.0,Ha


✓ Koreksi satuan Jamur tiram terverifikasi: jumlah kecamatan = nilai kabupaten = 2,948 M2.
--- Total_Produksi_Hortikultura_Buah.csv ---
  27 baris komoditas.
  ✓ Hasil akhir: 27 baris.



,komoditas,produksi_kuintal,satuan
0,Alpukat/Avocado,228170.92,Kw
1,Anggur/Grape,0.00,Kw
2,Apel/Apple,0.00,Kw
3,Belimbing/Star Fruit,44857.12,Kw
4,Duku/Langsat/Kokosan/Duku,43.00,Kw
5,Durian/Durian,202306.48,Kw
6,Jambu Air/Water Apple,21762.93,Kw
7,Jambu Biji/Guava,16287.51,Kw
8,Jengkol/Jengkol,115.50,Kw
9,Jeruk Besar/Pomelo,40.25,Kw


--- Total_Produksi_Hortikultura_Sayur.csv ---
  26 baris komoditas.
  ✓ Hasil akhir: 26 baris.



,komoditas,produksi_kuintal,satuan
0,Bawang Daun/Scallion,44264.00,Kw
1,Bawang Merah/Shallots,83602.00,Kw
2,Bawang Putih/Garlic,0.00,Kw
3,Bayam/Spinach,4390.00,Kw
4,Buncis/string bean,20940.00,Kw
5,Cabai Rawit/Chili/Cayenne Pepper,1095613.61,Kw
6,Kacang Panjang/Long Beans,47806.45,Kw
7,Kangkung/Water Spinach,4533.00,Kw
8,Kembang Kol/Cauliflower,23611.50,Kw
9,Kentang/Potato,0.00,Kw


In [11]:
# 11. CEK DOMINASI ANTAR KECAMATAN & VERIFIKASI NENAS NGANCAR (informasi, tidak mengubah data)

# Komoditas yang sebagian besar produksinya ada di satu kecamatan perlu diketahui sebelum clustering,
# karena kecamatan itu akan mendominasi jarak antar kecamatan jika tidak di-scale.
BATAS_DOMINASI = 0.5


def laporan_dominasi(df, kolom_nilai, judul):
    baris = []
    for komoditas, g in df.groupby("komoditas"):
        total = g[kolom_nilai].sum()
        if total <= 0:
            continue
        atas = g.loc[g[kolom_nilai].idxmax()]
        porsi = atas[kolom_nilai] / total
        if porsi >= BATAS_DOMINASI:
            baris.append((komoditas, atas["kecamatan"], porsi))
    print(f"{judul}: {len(baris)} komoditas dengan satu kecamatan >= {BATAS_DOMINASI:.0%} dari total kabupaten")
    for komoditas, kec, porsi in sorted(baris, key=lambda x: -x[2]):
        print(f"   {komoditas:<22} {kec:<12} {porsi:6.1%}")
    print()


laporan_dominasi(df_produksi_buah_kec, "produksi_kuintal", "Produksi buah")
laporan_dominasi(df_produksi_sayur_kec, "produksi_kuintal", "Produksi sayur")
laporan_dominasi(df_lp_sayur_kec, "luas_panen", "Luas panen sayur")

# Verifikasi Nenas di Ngancar: nilai di file bersih harus sama dengan total di file kabupaten.
nenas = df_produksi_buah_kec[df_produksi_buah_kec["komoditas"].str.lower().isin(["nenas", "nanas"])]
if nenas.empty:
    print("ℹ️  Komoditas Nenas tidak ditemukan di data buah.")
else:
    total_nenas = nenas["produksi_kuintal"].sum()
    total_buah = df_produksi_buah_kec["produksi_kuintal"].sum()
    ngancar = nenas.loc[nenas["kecamatan"] == "Ngancar", "produksi_kuintal"].iloc[0]
    print("Nenas di Ngancar:")
    print(f"   {ngancar:,.2f} kuintal = {ngancar / 10:,.0f} ton  (satuan file: KUINTAL; ton = kuintal / 10)")
    print(f"   = {ngancar / total_nenas:.1%} dari nenas kabupaten dan {ngancar / total_buah:.1%} dari seluruh produksi buah")

    kab_nenas = df_produksi_buah_kab[df_produksi_buah_kab["komoditas"].str.lower().str.contains("nenas|nanas")]
    if len(kab_nenas):
        sama = np.isclose(total_nenas, kab_nenas["produksi_kuintal"].iloc[0])
        print(f"   Jumlah 26 kecamatan {'=' if sama else '!='} nilai file kabupaten ({kab_nenas['produksi_kuintal'].iloc[0]:,.2f} kuintal)")

Produksi buah: 9 komoditas dengan satu kecamatan >= 50% dari total kabupaten
   Jengkol                Mojo         100.0%
   Nenas                  Ngancar       95.7%
   Manggis                Kandangan     86.5%
   Jeruk Besar            Gurah         84.5%
   Belimbing              Pagu          66.1%
   Duku/Langsat/Kokosan   Mojo          65.1%
   Jeruk Lemon            Plosoklaten   58.2%
   Melinjo                Badas         56.7%
   Salak                  Plosoklaten   54.2%

Produksi sayur: 4 komoditas dengan satu kecamatan >= 50% dari total kabupaten
   Bawang Daun            Kepung        93.5%
   Melon                  Kandat        76.3%
   Bayam                  Ngadiluwih    74.0%
   Cabai Keriting         Ringinrejo    53.1%

Luas panen sayur: 5 komoditas dengan satu kecamatan >= 50% dari total kabupaten
   Bawang Daun            Kepung        93.7%
   Bayam                  Ngadiluwih    80.0%
   Kangkung               Ngadiluwih    73.3%
   Melon                  K

## Ringkasan hal yang perlu ditindaklanjuti
1. **Jamur tiram** (luas panen sayur): satuan sudah dikoreksi ke M2 dan diverifikasi (cell 7 dan 10).
2. **Satuan campuran** (ha dan m² di luas panen sayur): jangan dijumlahkan lintas komoditas tanpa melihat
   kolom `satuan`. Untuk satu satuan, konversi m² ke hektar dengan dibagi 10.000.
3. **Nenas Ngancar** dan komoditas dominan lain (cell 11): bukan salah input, tapi perlu di-scale atau
   ditangani khusus saat clustering. Ingat satuan produksi buah adalah kuintal.
4. Semua angka "-" dianggap **0** (konvensi BPS). Kalau asumsi ini salah untuk kasusmu, ubah `parse_angka()`
   di cell 4.
5. File kabupaten (`*_kabupaten.csv`) belum dicocokkan nama komoditasnya satu-satu dengan file per kecamatan
   (mis. "Bawang Daun" vs "Bawang Daun/Scallion"). Itu perlu tabel master komoditas terpisah.
6. Tidak ada kolom tahun di file sumber. Catat tahun datanya, karena dibutuhkan untuk deret waktu.
